# 06 - Optional targeted review setup

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

**Run order:** base setup -> scoped FAR test -> this notebook -> targeted parent. Set the basic parameters below and Run all to opt in. Keep the **FAR output Lakehouse** attached and leave stamped advanced wiring unchanged.

## Selection

- `RANKING_METRIC`: `cu_seconds` (default) or `recorded_throttling_minutes` (recorded minutes can be incomplete; they are not operation counts).
- `TOP_N`: 1-100 workspaces, default five. `LOOKBACK_DAYS`: 1-28 complete UTC dates before today, not schedule frequency.
- `CAPACITY_ID_OR_NAME`: blank for all capacities, one GUID/exact case-insensitive FUAM capacity name, or a JSON list of names/GUIDs. For a set, use `CAPACITY_ID_OR_NAME = '["Capacity A", "Capacity B"]'`. `TOP_N` ranks workspaces across the combined set, not per capacity. Any invalid or ambiguous entry fails the selection; `[]` is not all capacities.
- Set `FUAM_LAKEHOUSE_ID` explicitly if the source workspace has multiple Lakehouses.

Success prints the **parent pipeline**, **Runner** and **Completion** IDs; the original FAR pipeline is unchanged. Run reviews through the parent, not the helper notebooks. Empty selection skips FAR and email.

Test with a parent `WORKSPACE_IDS` candidate allow-list and notifications off. Confirm selected IDs, child success and the report. The parent does not inherit standalone FAR's saved scope. Leave the allow-list blank for normal automatic ranking. Schedule only the validated parent for targeted reviews; keep the independent daily 07 sync for owner reporting. Setup enables neither schedule.

## Optional email and owner access

Pause schedules. On the parent canvas, open **Send owner emails -> Email workspace administrator**. In the Outlook activity's **Settings**, create/select a connection and sign in as the sender. Keep generated **To/Subject/Body**, set **General -> Activity state -> Activated**, save and configure `FAR_REPORT_URL`. Then test `NOTIFICATIONS_ENABLED="true"`. The activity starts inactive and must be connected and activated first.

Use the same account for connection creation and the first test run. A test workspace must have positive FUAM metrics; choose one where you are its sole direct-user administrator to send only to yourself. Other eligible administrators also receive mail. Setting notifications off skips email, not owner-access sync. Verify delivery in native monitoring and the mailbox; inspect uncertain outcomes before retrying to avoid duplicates.

06 does not create owner reporting or grant access. For owners, use the separately deployed and validated owner report, with a fixed-identity connection, source SSO disabled, successful 07 sync, item-scoped Read and `WorkspaceOwner` membership. Never grant FAR workspace roles (including Viewer), raw source access or Build as part of owner approval. Governance links and URL filters are not security boundaries. Grants expire after 24 hours; run daily sync without overlapping targeted/manual syncs. Disabling deployment does not revoke existing sharing.

## Redeployment

Pause schedules and finish active runs before rerunning all cells. Email connection, From, activation state and report URL are preserved; the notification default resets to false and generated bindings are restored. Inspect schedule overrides before resuming.

See [deployment](https://github.com/microsoft/fabric-architecture-review/blob/main/fabric/DEPLOYMENT.md), [targeted-review reference](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/targeted-review.md) and [owner access](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/workspace-owner-report.md) for the complete operating requirements. Restrict monitoring access; outputs can contain contact metadata.


In [ ]:
# Copyright (c) Microsoft Corporation.
# Licensed under the MIT License.
# Basic settings
DEPLOY_TARGETED_REVIEW = "false"
FUAM_WORKSPACE_ID = ""
FUAM_LAKEHOUSE_ID = ""
CAPACITY_ID_OR_NAME = ""
RANKING_METRIC = "cu_seconds"
LOOKBACK_DAYS = "7"
TOP_N = "5"

# Advanced wiring - automatically populated by FAR setup
GITHUB_REPO_URL = "https://github.com/microsoft/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
GITHUB_REF = ""
WORKSPACE_ID = ""
LAKEHOUSE_ID = ""
CHILD_PIPELINE_ID = ""
OWNER_ACCESS_NOTEBOOK_ID = ""
PARENT_PIPELINE_NAME = "Fabric Arch Review - Targeted Review"


In [ ]:
import os, sys, shutil, subprocess
import tempfile

def _flag(value):
    if str(value).lower() not in ("true", "false"):
        raise ValueError("Feature switches must be true or false.")
    return str(value).lower() == "true"

if not _flag(DEPLOY_TARGETED_REVIEW):
    print("Targeted review is disabled. No optional artifacts or notifications were created.")
else:
    WORK_ROOT = tempfile.mkdtemp(prefix="far-targeted-setup-")
    REPO_DIR = os.path.join(WORK_ROOT, "repo")
    _url = GITHUB_REPO_URL
    _ref = (GITHUB_REF or "").strip() or GITHUB_BRANCH
    subprocess.run(["git", "clone", "--branch", _ref, "--depth", "1", _url, REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)


In [ ]:
if _flag(DEPLOY_TARGETED_REVIEW):
    import json
    from pathlib import Path
    import notebookutils
    for module in [key for key in sys.modules if key == "orchestration" or key.startswith("orchestration.")]:
        del sys.modules[module]
    from orchestration.setup import provision
    from orchestration.fabric_api import FabricClient
    client = FabricClient(lambda: notebookutils.credentials.getToken("pbi"))
    result = provision(
        client, workspace_id=WORKSPACE_ID, lakehouse_id=LAKEHOUSE_ID,
        child_pipeline_id=CHILD_PIPELINE_ID, repo_dir=Path(REPO_DIR),
        owner_access_notebook_id=OWNER_ACCESS_NOTEBOOK_ID,
        name=PARENT_PIPELINE_NAME,
        defaults={
            "CAPACITY_ID_OR_NAME": CAPACITY_ID_OR_NAME,
            "FUAM_WORKSPACE_ID": FUAM_WORKSPACE_ID,
            "FUAM_LAKEHOUSE_ID": FUAM_LAKEHOUSE_ID,
            "RANKING_METRIC": RANKING_METRIC, "LOOKBACK_DAYS": LOOKBACK_DAYS,
            "TOP_N": TOP_N,
        },
    )
    from orchestration.folders import organize
    organize(client, WORKSPACE_ID, [
        (result['pipeline_id'], 'DataPipeline', 'Pipelines'),
        (CHILD_PIPELINE_ID, 'DataPipeline', 'Pipelines'),
        (result['notebook_id'], 'Notebook', 'Notebooks'),
        (result['completion_notebook_id'], 'Notebook', 'Notebooks'),
        (LAKEHOUSE_ID, 'Lakehouse', None),
    ])
    _ctx = notebookutils.runtime.context
    organize(client, _ctx.get('currentWorkspaceId') or _ctx.get('workspaceId'),
             [(_ctx['currentNotebookId'], 'Notebook', 'Notebooks')])
    print(json.dumps(result, indent=2))
    print("Run the parent pipeline manually first, then configure its schedule in Fabric.")
    print("Owner emails default to off. On the parent's main canvas, open Send owner emails -> Email workspace administrator. Connect and activate the email activity, set FAR_REPORT_URL, then test with NOTIFICATIONS_ENABLED=true. Existing mail settings are preserved; check schedule overrides.")
